# 48 — Midterm report figures and tables

Generates the two visuals adopted for the midterm report (no GPU; all
inputs are cached artifacts in `resources/`):

1. **Guide-validation histogram** — validated singles Δ over the full
   161-concept slate (nb32), stacked so the original 27 method-development
   concepts are visible against the rest, with the pool floor marked.
   → `resources/48_slate_histogram.png`
2. **Cross-model table (blend held fixed)** — both-improved rates from the
   validated-pool cross-model run (nb43) plus the English-only
   lemma-policy control row (nb46). → `resources/48_crossmodel_table.csv`

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
pool_d = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
pool = {e["synset"]: e["delta"] for e in pool_d["pool"]}
slate = {**pool, **{e["synset"]: e["delta"] for e in pool_d["rejected"]}}
floor = min(pool.values())

pairs = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())
legacy = sorted({s for p in pairs["all_pairs"] for s in (p["a"], p["b"])})
survivors = [s for s in legacy if s in pool]
assert set(legacy) <= set(slate)
print(f"slate {len(slate)}, floor {floor:.2f}, legacy {len(legacy)}, survive {survivors}")

rest = [d for s, d in slate.items() if s not in legacy]
leg_d = [slate[s] for s in legacy]

fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.hist(
    [rest, leg_d],
    bins=32,
    stacked=True,
    color=["lightsteelblue", "tab:red"],
    edgecolor="white",
    label=[f"validated slate (n={len(slate)})", "original 27 concepts"],
)
ax.axvline(floor, color="black", ls="--", lw=1.2)
top = ax.get_ylim()[1]
ax.annotate(f"pool floor ({floor:.2f})", xy=(floor, top * 0.95),
            xytext=(floor + 0.04, top * 0.84), fontsize=8)
ax.annotate(f"only {len(survivors)} of {len(legacy)} survive", xy=(1.22, 2),
            xytext=(1.42, top * 0.5), fontsize=8, color="tab:red",
            arrowprops=dict(arrowstyle="->", color="tab:red", lw=0.8))
ax.set_xlabel("validated single-concept expression \u0394 (bf16 Llama, greedy)")
ax.set_ylabel("concepts")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.savefig("resources/48_slate_histogram.png", dpi=170)
plt.show()
print(f"legacy \u0394 range {min(leg_d):.2f}\u2013{max(leg_d):.2f} "
      f"(\u2248{max(leg_d)/min(leg_d):.0f}\u00d7 spread), "
      f"slate range {min(slate.values()):.2f}\u2013{max(slate.values()):.2f}")

In [ ]:
s43 = pd.read_csv("resources/43_validated_pool_summary.csv")
s46 = pd.read_csv("resources/46_lemma_policy_summary.csv")

order = ["llama-3.1-8b-awq", "llama-3.1-8b-bf16", "gemma-2-2b", "gemma-2-9b"]
blend = (
    s43[s43.method == "blend"]
    .set_index("model")
    .loc[order]
    .reset_index()[["model", "both_improved", "both_improved_cos"]]
)
ctrl = s46[(s46.policy == "english-only") & (s46.method == "blend")]
ctrl_row = pd.DataFrame(
    [{
        "model": "llama-3.1-8b-bf16 (english-only control)",
        "both_improved": ctrl.both_improved.iloc[0],
        "both_improved_cos": ctrl.both_improved_cos.iloc[0],
    }]
)
table = pd.concat([blend.iloc[:2], ctrl_row, blend.iloc[2:]]).reset_index(drop=True)
table.to_csv("resources/48_crossmodel_table.csv", index=False)
print(table.round(3).to_string(index=False))